In [1]:
# NOTEBOOK 39 - FINAL MODEL: the wind-aware graph model (dyn_adv) scored and documented for the report and the live system. NO training.
# dyn_adv = the 5 checkpoints watgcn_v5dyn_adv_seed*.pt from notebook 28: WA-TGCN with the hourly wind graph A(t) + the upwind transport feature (28 input channels).
# Two settings are scored on the Delhi test period (8 Dec 2025 - 9 Apr 2026), same 2,918 windows:
#   forecast : the 24 future hours get the REAL day-old forecast wind and forecast-wind graph  -> the honest, deployable score (expected 58.32)
#   perfect  : the 24 future hours get the actual (ERA5) future wind                           -> the ceiling (expected 57.71, notebook 28)
# New files only (nothing existing is overwritten; the older final_metrics.json / final_per_station.csv belong to v1 and are left alone):
#   results/dynadv_final_metrics.json, results/dynadv_per_station.csv, results/dynadv_test_pred_forecast.npy, results/dynadv_test_starts.npy,
#   models/dynadv_final_config.json, scalers/v5_feature_scaler.pkl (the 30-channel scaler the live system needs)
# HOW TO CHECK: this cell only prints a note. The next cells rebuild the data (about 2 minutes).
print('final model evaluation: dyn_adv, 5 seeds, forecast-wind and perfect-wind settings')

final model evaluation: dyn_adv, 5 seeds, forecast-wind and perfect-wind settings


In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
PREVDAY_NPY     = os.path.join(V2_DIR, 'openmeteo_prevday1_uv.npy')
ADJ_FC_NPY      = os.path.join(V2_DIR, 'dynamic_adj_prevday1.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [3]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')
prevday = np.load(PREVDAY_NPY).astype(np.float32)                    # day-old forecast wind (T,N,2), blowing TOWARD
adj_fc = np.load(ADJ_FC_NPY).astype(np.float32)                      # forecast-wind graph for the future hours (T,N,N)
assert prevday.shape == era5.shape and adj_fc.shape == dyn_adj.shape, (prevday.shape, adj_fc.shape)
print('forecast wind', prevday.shape, '| forecast graph', adj_fc.shape, '| mean |forecast - ERA5| wind speed difference (m/s):',
      float(np.abs(np.hypot(prevday[..., 0], prevday[..., 1]) - era5_spd).mean()))

T=19921  dyn_adj(ERA5) (19921, 37, 37)  static edges 508
forecast wind (19921, 37, 2) | forecast graph (19921, 37, 37) | mean |forecast - ERA5| wind speed difference (m/s): 0.7865347862243652


In [4]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_21600\1524811802.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month


F=26  NaNs 0


In [5]:
# 1) geometry and the two new feature pairs for Delhi
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
THRESHOLD_KM, MAX_LAG = 12.0, 6

def geometry(lats, lons):
    n = len(lats)
    dist = np.zeros((n, n)); du = np.zeros((n, n)); dv = np.zeros((n, n))
    R = 6371.0
    for i in range(n):
        for j in range(n):
            if i != j:
                p1, p2 = np.radians(lats[i]), np.radians(lats[j])
                a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lons[j] - lons[i]) / 2) ** 2
                dist[i, j] = 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
                de = (lons[j] - lons[i]) * np.cos(np.radians(lats[i])); dn = lats[j] - lats[i]; nr = np.hypot(de, dn)
                if nr > 0:
                    du[i, j], dv[i, j] = de / nr, dn / nr          # unit vector from i to j
    return dist, du, dv, (dist > 0) & (dist <= THRESHOLD_KM)

def transport_features(pm, uv, dist, du, dv, within):
    # pm (T,n) raw PM2.5, uv (T,n,2) ERA5 wind (blowing TOWARD). Only values at or before hour t are used (no leakage).
    T, n = pm.shape
    adv = pm.copy(); nb = pm.copy()
    src = np.arange(n)[:, None] * np.ones((1, n), dtype=int)           # source station index j for entry [j, i]
    stat = within.astype(float); cnt = stat.sum(0); has_nb = cnt > 0
    for t in range(T):
        align = uv[t, :, 0][:, None] * du + uv[t, :, 1][:, None] * dv   # wind at j along the direction j -> i  (m/s)
        w = np.where(within & (align > 0), align / (dist + 1e-6), 0.0)  # j is upwind of i
        wsum = w.sum(0); has = wsum > 0
        lag = np.clip(np.rint(dist / (np.maximum(align, 0.5) * 3.6)), 0, MAX_LAG).astype(int)   # hours of travel (km / (km/h))
        vals = pm[np.clip(t - lag, 0, T - 1), src]                      # pm[t - lag[j,i], j]
        adv[t, has] = ((w * vals).sum(0))[has] / wsum[has]
        nb[t, has_nb] = ((stat * pm[t][:, None]).sum(0))[has_nb] / cnt[has_nb]
    return adv, nb

dist, du, dv, within = geometry(lats, lons)
t0 = time.time()
adv, nb = transport_features(X_raw[:, :, 0].astype(float), era5, dist, du, dv, within)
pm0 = X_raw[:, :, 0]
extra = np.stack([adv - 0, adv - pm0, nb - 0, nb - pm0], axis=2).astype(np.float32)       # adv_pm, adv_diff, nb_pm, nb_diff
extra[:, :, 0] = adv; extra[:, :, 2] = nb
X30 = np.concatenate([X_raw, extra], axis=2)
print(f'features built in {time.time()-t0:.0f}s | X30 {X30.shape}')
print('mean |adv - own| :', float(np.abs(adv - pm0).mean()), '| mean |nb - own| :', float(np.abs(nb - pm0).mean()))
print('corr(adv, own PM):', float(np.corrcoef(adv.ravel(), pm0.ravel())[0, 1]), '| corr(nb, own PM):', float(np.corrcoef(nb.ravel(), pm0.ravel())[0, 1]))

# HOW TO CHECK: about a minute. Both correlations are high (0.8-0.95) because neighbouring stations resemble each other;
# the question for the models is whether the lagged, wind-aligned version adds anything beyond the same-hour mean.

features built in 1s | X30 (19921, 37, 30)
mean |adv - own| : 28.104649143919474 | mean |nb - own| : 23.36611597159713
corr(adv, own PM): 0.8577609401778713 | corr(nb, own PM): 0.9041709851392609


In [6]:
# 2) scaler for the 30 channels (fit on the training period only); the first 26 must match the saved v3 scaler
from sklearn.preprocessing import StandardScaler
import pickle
T = X30.shape[0]; tr_end = int(T * 0.70)
sc30 = StandardScaler().fit(X30[:tr_end].reshape(-1, 30))
v3s = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
assert np.allclose(sc30.mean_[:26], v3s.mean_, rtol=1e-4, atol=1e-4) and np.allclose(sc30.scale_[:26], v3s.scale_, rtol=1e-4, atol=1e-4), 'scaler mismatch'
log_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Xs_full = sc30.transform(X30.reshape(-1, 30)).reshape(T, N, 30).astype(np.float32)
Yl_full = to_log(X30[:, :, 0])
Y_raw = X30[:, :, 0].copy()
print('scaler check passed: the first 26 channels match the v3 scaler')

# HOW TO CHECK: prints the pass line. An assertion error means the data or train split changed - stop and tell me.

scaler check passed: the first 26 channels match the v3 scaler


C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# future-hour wind channels from the day-old forecast (scaled with the same train-period scaler as the observed wind)
pv_spd = np.hypot(prevday[..., 0], prevday[..., 1])
pv_raw = np.stack([prevday[..., 0], prevday[..., 1], pv_spd], axis=2).astype(np.float32)          # (T,N,3)
WIDX = [U_IDX, V_IDX, W_IDX]
Xs_fut = Xs_full.copy()
Xs_fut[:, :, WIDX] = ((pv_raw - sc30.mean_[WIDX]) / sc30.scale_[WIDX]).astype(np.float32)
print('future wind channels now come from the day-old forecast | scaled means: observed', Xs_full[:, :, WIDX].mean((0, 1)).round(2), 'forecast', Xs_fut[:, :, WIDX].mean((0, 1)).round(2))

# HOW TO CHECK: the u and v means are close; the forecast speed mean is lower than the observed one (about -0.5 against -0.1 on the scaled axis) because the day-old forecast runs slower than ERA5. That bias is real and is why the models are retrained on it.

future wind channels now come from the day-old forecast | scaled means: observed [ 0.04  0.01 -0.13] forecast [-0.07  0.14 -0.5 ]


In [8]:
# model with three graph modes plus the learned blend, and optional context dropout at training time
CTX_COLS = ['city_mean_pm', 'city_max_pm', 'pm_minus_city']
CTX_IDX = [FEATURE_COLS.index(c) for c in CTX_COLS]
WSPD_IDX = FEATURE_COLS.index('wspd')
FUT_WSPD = 9                                  # wspd is the last of the 10 future channels (7 calendar + u, v, wspd)

class WATGCNv4(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon, n_stations,
                 dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        sa = sa.clone(); sa.fill_diagonal_(0)
        self.register_buffer('static_adj', sa, persistent=False)
        self.register_buffer('static_rn', sa / sa.sum(1, keepdim=True).clamp(min=1.0), persistent=False)   # row-normalised
        if graph_mode == 'blend':
            self.blend_a = nn.Parameter(torch.zeros(1)); self.blend_b = nn.Parameter(torch.zeros(1))

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def alpha(self, speed):                       # speed: (B, steps) standardised wind speed -> weight of the wind graph
        return torch.sigmoid(self.blend_a + self.blend_b * speed)[..., None, None]

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            A = None
        elif self.graph_mode == 'dynamic':
            A = self._n(At + I.view(1, 1, n, n))
        elif self.graph_mode == 'static':
            A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
        else:                                      # blend
            a = self.alpha(X[..., WSPD_IDX].mean(-1))
            A = self._n(a * At + (1 - a) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(h if A is None else torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        elif self.graph_mode == 'static':
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        else:
            af = self.alpha(Fut[..., FUT_WSPD].mean(-1))
            ctx_fc = torch.matmul(self._n(af * Afc + (1 - af) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n)), ctx_rep)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, self.fut_enc(Fut)], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [9]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

In [10]:
# score the 5 dyn_adv checkpoints in both settings
SEEDS = [42, 7, 123, 2024, 999]
ADV_COLS = list(range(26)) + [26, 27]                         # 26 base channels + adv_pm + adv_diff
STATIC_NP = np.load(STATIC_ADJ_NPY).astype(np.float32)
val_end = int(T * 0.85)
def split_starts(lo, hi): return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
STARTS = {'val': split_starts(tr_end, val_end), 'test': split_starts(val_end, T)}
print(f"windows -- val {len(STARTS['val']):,} | test {len(STARTS['test']):,}")

class DSE(Dataset):
    def __init__(self, starts, Xfut, Afc): self.s, self.Xfut, self.Afc = np.asarray(starts, np.int64), Xfut, Afc
    def __len__(self): return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(np.ascontiguousarray(Xs_full[g:i1][:, :, ADV_COLS])), torch.from_numpy(dyn_adj[g:i1].copy()),
                torch.from_numpy(self.Afc[i1:i2].copy()), torch.from_numpy(np.ascontiguousarray(self.Xfut[i1:i2][:, :, FUT_IDX])),
                torch.from_numpy(Yl_full[i1 - 1].copy()), torch.from_numpy(Yl_full[i1:i2].copy()), torch.tensor(1.0))
SETTING = {'forecast': (Xs_fut, adj_fc), 'perfect': (Xs_full, dyn_adj)}

P, ACT = {}, {}
for setting, (xf, af) in SETTING.items():
    for split in ('val', 'test'):
        ld = DataLoader(DSE(STARTS[split], xf, af), batch_size=BATCH_SIZE, shuffle=False, num_workers=0); ps = []
        for sd in SEEDS:
            m = WATGCNv4(len(ADV_COLS), HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, N_FUT, GCN_HOPS, graph_mode='dynamic', static_adj=STATIC_NP).to(DEVICE)
            m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v5dyn_adv_seed{sd}.pt'), map_location=DEVICE))
            p, a = predict(m, ld); ps.append(p); del m
        P[(setting, split)] = ps; ACT[split] = a
        print(f'{setting:9} {split:5} scored {len(SEEDS)} seeds')

ens = lambda setting, split: np.mean(P[(setting, split)], axis=0)
RES = {s: {sp: sc(ens(s, sp), ACT[sp]) for sp in ('val', 'test')} for s in SETTING}
print(f"\n{'setting':10}{'split':7}{'RMSE':>8}{'MAE':>8}{'R2':>8}")
for s in SETTING:
    for sp in ('val', 'test'): print(f"{s:10}{sp:7}{RES[s][sp]['rmse']:>8.2f}{RES[s][sp]['mae']:>8.2f}{RES[s][sp]['r2']:>8.3f}")
print(f"\nmatches notebook 37 (58.32): {abs(RES['forecast']['test']['rmse'] - 58.32) < 0.02} | matches notebook 28 (57.71): {abs(RES['perfect']['test']['rmse'] - 57.71) < 0.02}")

# HOW TO CHECK: forecast/test is about 58.32 and perfect/test about 57.71; both lines above say True. False means the data or checkpoints differ from what produced those numbers - stop and tell me.

windows -- val 2,917 | test 2,918
forecast  val   scored 5 seeds
forecast  test  scored 5 seeds
perfect   val   scored 5 seeds
perfect   test  scored 5 seeds

setting   split      RMSE     MAE      R2
forecast  val       55.05   30.11   0.739
forecast  test      58.32   38.21   0.665
perfect   val       54.95   29.98   0.740
perfect   test      57.71   37.92   0.672

matches notebook 37 (58.32): True | matches notebook 28 (57.71): True


In [11]:
# breakdowns for the report (the forecast-wind setting is the headline; the perfect-wind numbers are kept alongside)
ar = from_log(ACT['test']); anc = np.stack([Yl_full[g + INPUT_WINDOW - 1] for g in STARTS['test']])
pers = from_log(np.repeat(anc[:, None, :], HORIZON, axis=1))
pf, pp = from_log(ens('forecast', 'test')), from_log(ens('perfect', 'test'))
rm = lambda p, a: float(np.sqrt(np.mean((p - a) ** 2)))
hz = lambda p: {f'+{h + 1}h': rm(p[:, h], ar[:, h]) for h in range(HORIZON)}
BY_H = {'persistence': hz(pers), 'forecast': hz(pf), 'perfect': hz(pp)}
print(f"{'RMSE by horizon':18}" + ''.join(f'{h:>8}' for h in ('+1h', '+3h', '+6h', '+12h', '+18h', '+24h')))
for k in BY_H: print(f'{k:18}' + ''.join(f'{BY_H[k][h]:>8.1f}' for h in ('+1h', '+3h', '+6h', '+12h', '+18h', '+24h')))

sw = ar.max(1) - ar.min(1); q33, q66 = np.percentile(sw, [33, 66])
VREG = {'Stable': sw <= q33, 'Moderate': (sw > q33) & (sw <= q66), 'Volatile': sw > q66}
def by_reg(p, regs): return {k: rm(p.transpose(0, 2, 1)[m], ar.transpose(0, 2, 1)[m]) for k, m in regs.items()}
BY_VOL = {'persistence': by_reg(pers, VREG), 'forecast': by_reg(pf, VREG), 'perfect': by_reg(pp, VREG)}
ws_win = np.array([np.hypot(era5[g + INPUT_WINDOW: g + INPUT_WINDOW + HORIZON, :, 0], era5[g + INPUT_WINDOW: g + INPUT_WINDOW + HORIZON, :, 1]).mean() for g in STARTS['test']])
w33, w66 = np.percentile(ws_win, [33, 66])
WREG_MASK = {'Calm': ws_win <= w33, 'Breezy': (ws_win > w33) & (ws_win <= w66), 'Windy': ws_win > w66}
def by_win(p): return {k: rm(p[m], ar[m]) for k, m in WREG_MASK.items()}
BY_WIND = {'persistence': by_win(pers), 'forecast': by_win(pf), 'perfect': by_win(pp)}
print(f"\n{'by volatility':18}" + ''.join(f'{k:>10}' for k in VREG))
for k in BY_VOL: print(f'{k:18}' + ''.join(f'{BY_VOL[k][r]:>10.1f}' for r in VREG))
print(f"\n{'by forecast wind':18}" + ''.join(f'{k:>10}' for k in WREG_MASK) + f'   (tertile cut-offs {w33:.2f} / {w66:.2f} m/s)')
for k in BY_WIND: print(f'{k:18}' + ''.join(f'{BY_WIND[k][r]:>10.1f}' for r in WREG_MASK))

seed_rmse = [sc(p, ACT['test'])['rmse'] for p in P[('forecast', 'test')]]
print(f'\nper-seed test RMSE (forecast wind): {[round(x, 2) for x in seed_rmse]} | mean {np.mean(seed_rmse):.2f} | std {np.std(seed_rmse):.2f}')

rows = []
for i, nm in enumerate(station_names):
    a_, p_, q_ = ar[:, :, i].ravel(), pf[:, :, i].ravel(), pers[:, :, i].ravel()
    rows.append({'station': nm, 'rmse': rm(p_, a_), 'mae': float(np.mean(np.abs(p_ - a_))), 'r2': float(1 - np.sum((p_ - a_) ** 2) / np.sum((a_ - a_.mean()) ** 2)),
                 'persistence_rmse': rm(q_, a_), 'mean_pm25_test': float(a_.mean())})
PS = pd.DataFrame(rows)
print(f"per-station test RMSE: mean {PS['rmse'].mean():.1f} | best {PS.loc[PS['rmse'].idxmin(), 'station']} {PS['rmse'].min():.1f} | worst {PS.loc[PS['rmse'].idxmax(), 'station']} {PS['rmse'].max():.1f}"
      f" | stations better than persistence: {int((PS['rmse'] < PS['persistence_rmse']).sum())} of {N}")

# HOW TO READ: persistence wins only at +1h; the forecast setting is within about 0.6 of the perfect-wind setting at every horizon; the volatile third holds most of the error.

RMSE by horizon        +1h     +3h     +6h    +12h    +18h    +24h
persistence           33.8    61.2    82.9    93.2    92.6    76.7
forecast              41.5    48.7    55.3    59.8    62.4    63.0
perfect               42.3    48.5    54.8    59.0    61.7    62.3

by volatility         Stable  Moderate  Volatile
persistence             37.0      62.9     124.6
forecast                29.1      41.5      86.6
perfect                 28.9      41.3      85.6

by forecast wind        Calm    Breezy     Windy   (tertile cut-offs 1.44 / 1.90 m/s)
persistence            105.3      76.9      64.8
forecast                78.5      49.3      40.8
perfect                 77.5      48.8      40.5

per-seed test RMSE (forecast wind): [59.84, 61.39, 58.69, 59.79, 59.85] | mean 59.91 | std 0.86
per-station test RMSE: mean 57.2 | best IGI Airport (T3) 37.2 | worst Anand Vihar 79.5 | stations better than persistence: 37 of 37


In [12]:
# save the files the live system and the front end need (new files only)
OUT_RES, OUT_MOD, OUT_SCL = RESULTS_DIR, MODEL_DIR, SCALER_DIR
PS.round(4).to_csv(os.path.join(OUT_RES, 'dynadv_per_station.csv'), index=False)
np.save(os.path.join(OUT_RES, 'dynadv_test_pred_forecast.npy'), pf.astype(np.float32))
np.save(os.path.join(OUT_RES, 'dynadv_test_starts.npy'), STARTS['test'])
pickle.dump(sc30, open(os.path.join(OUT_SCL, 'v5_feature_scaler.pkl'), 'wb'))
json.dump({'model': 'WA-TGCN dyn_adv (wind-aware hourly graph + upwind transport feature)', 'seeds': SEEDS,
           'checkpoints': [f'watgcn_v5dyn_adv_seed{s}.pt' for s in SEEDS], 'graph_mode': 'dynamic', 'in_features': len(ADV_COLS),
           'feature_cols': FEATURE_COLS + ['adv_pm', 'adv_diff'], 'scaler_30ch': 'scalers/v5_feature_scaler.pkl (30 channels; columns 0-27 are used by this model)',
           'log_scaler': 'scalers/v3_log_scaler.pkl', 'hidden_dim': HIDDEN_DIM, 'gru_layers': GRU_LAYERS, 'attn_heads': ATTN_HEADS, 'dropout': DROPOUT, 'gcn_hops': GCN_HOPS,
           'input_window': INPUT_WINDOW, 'horizon': HORIZON, 'n_stations': N, 'n_fut': N_FUT, 'future_channels': ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend', 'u_wind', 'v_wind', 'wspd'],
           'transport_feature': {'threshold_km': THRESHOLD_KM, 'max_lag_hours': MAX_LAG, 'wind': 'observed ERA5 wind at the source station', 'definition': 'adv = alignment/distance weighted mean of upwind neighbours PM2.5 at lag = round(dist / (max(align, 0.5) * 3.6)) capped at 6 h; adv_diff = adv - own PM2.5'},
           'future_wind_live': 'real day-old forecast wind (Open-Meteo previous-day-1 / forecast API) and forecast-wind graph', 'loss_vol_alpha': LOSS_VOL_ALPHA,
           'split': {'train_end_index': tr_end, 'val_end_index': val_end, 'test_start': str(timestamps[val_end]), 'test_windows': int(len(STARTS['test']))}},
          open(os.path.join(OUT_MOD, 'dynadv_final_config.json'), 'w'), indent=2)
json.dump({'model': 'WA-TGCN dyn_adv, 5-seed ensemble', 'headline_setting': 'forecast (real day-old forecast wind)', 'overall': RES,
           'persistence': sc(np.repeat(anc[:, None, :], HORIZON, axis=1), ACT['test']), 'by_horizon': BY_H, 'by_volatility': BY_VOL, 'by_forecast_wind_regime': BY_WIND,
           'per_seed_test_rmse_forecast': seed_rmse, 'regime_cutoffs': {'swing_q33': float(q33), 'swing_q66': float(q66), 'wind_q33': float(w33), 'wind_q66': float(w66)},
           'note': 'forecast = real day-old forecast wind and graph for the 24 future hours; perfect = ERA5 future wind (ceiling)'},
          open(os.path.join(OUT_RES, 'dynadv_final_metrics.json'), 'w'), indent=2)
print('saved: dynadv_final_metrics.json, dynadv_per_station.csv, dynadv_test_pred_forecast.npy, dynadv_test_starts.npy, dynadv_final_config.json, v5_feature_scaler.pkl')

# HOW TO CHECK: the line above lists six saved files. Existing final_metrics.json and final_per_station.csv (v1) are not touched.

saved: dynadv_final_metrics.json, dynadv_per_station.csv, dynadv_test_pred_forecast.npy, dynadv_test_starts.npy, dynadv_final_config.json, v5_feature_scaler.pkl
